In [ ]:
"""
Step 3a - Data Profiling (see and understand the Bronze data)
Customer Health & Sentiment Scoring Multi-Agent AI Solution

Read-only. Nothing in Bronze is changed.

Outputs (reports/profile/):
  01_overview.csv              rows, PK uniqueness, duplicates, customers per table
  02_schema_<table>.csv        column names and types
  03_summary_<table>.csv       min / max / mean / std / quartiles / approx distinct / null %
  04_distinct_values.csv       value counts for every categorical column (raw + normalised)
  05_time_coverage.csv         min/max timestamp, out-of-window rows, month-vs-timestamp mismatches
  06_rows_by_month.csv         event volume and active customers per month per table
  07_fk_coverage.csv           orphan customer_ids in each event table
  08_range_preview.csv         how many rows would break each planned range rule
  09_events_per_customer.csv   distribution of events per customer per table
  10_label_profile.csv         sentiment_label vs sentiment_score_gt (ground truth sanity)
  11_samples_<table>.csv       20 random rows per table
  config/allowed_values_suggested.json   -> review, then save as config/allowed_values.json

Run:  python step3a_profile.py            (or: python run_pipeline.py --only 3a)

Change log (see decisions.md): D05 range preview reads business_rules.RANGES (same limits Step 3b enforces),
D09 notebook-safe finish(), section markers for the .ipynb version.
"""

## 1. Imports and configuration

In [ ]:
from __future__ import annotations
import json

from common import (ALLOWED_VALUES_SUGGESTED, BRONZE_DIR, CATEGORICAL, CONFIG_DIR,
                    EVENT_TABLES, EVENT_TS, LINEAGE_COLUMNS, PK, REPORT_DIR, TABLES,
                    WINDOW_END, WINDOW_START, copy_to_csv, get_connection, log, norm,
                    present_columns, register_layer, finish)
from business_rules import ENG_LONG_SESSION_WARN_SEC, RANGES

OUT = REPORT_DIR / "profile"
MAX_DISTINCT_FOR_ALLOWED_LIST = 100   # columns with more values are not treated as enums

## 2. Helpers

In [ ]:
def columns_of(con, view: str) -> list[tuple[str, str]]:
    return [(r[0], r[1]) for r in con.execute(f"DESCRIBE SELECT * FROM {view}").fetchall()]


def section(title: str) -> None:
    print("\n" + "=" * 100 + f"\n{title}\n" + "=" * 100)


# ---------------------------------------------------------------------------

## 3. Profiling sections

In [ ]:
def overview(con) -> None:
    section("1. OVERVIEW - rows, keys, duplicates")
    parts = []
    for t in TABLES:
        v = f"v_bronze_{t}"
        biz_cols = [c for c, _ in columns_of(con, v) if c not in LINEAGE_COLUMNS]
        no_pk_cols = [c for c in biz_cols if c != PK[t]]
        parts.append(f"""
            SELECT '{t}' AS table_name,
                   (SELECT count(*) FROM {v})                                        AS row_count,
                   {len(biz_cols)}                                                    AS column_count,
                   (SELECT count(DISTINCT {PK[t]}) FROM {v})                          AS distinct_pk,
                   (SELECT count(*) FROM {v} WHERE {PK[t]} IS NULL)                   AS null_pk,
                   (SELECT count(*) FROM (SELECT {PK[t]} FROM {v} GROUP BY 1
                                          HAVING count(*) > 1))                       AS duplicated_pk_values,
                   (SELECT count(*) FROM {v}) -
                   (SELECT count(*) FROM (SELECT DISTINCT {", ".join(biz_cols)} FROM {v})) AS exact_duplicate_rows,
                   (SELECT count(*) FROM {v}) -
                   (SELECT count(*) FROM (SELECT DISTINCT {", ".join(no_pk_cols)} FROM {v})) AS same_content_diff_pk,
                   (SELECT count(DISTINCT customer_id) FROM {v})                     AS distinct_customers
        """)
    q = " UNION ALL ".join(parts)
    con.sql(q).show(max_width=200)
    copy_to_csv(con, q, OUT / "01_overview.csv")


def schema_and_summary(con) -> None:
    section("2-3. SCHEMA AND COLUMN SUMMARY (min/max/mean/quartiles/null %)")
    for t in TABLES:
        v = f"v_bronze_{t}"
        copy_to_csv(con, f"DESCRIBE SELECT * FROM {v}", OUT / f"02_schema_{t}.csv")
        q = f"SELECT * FROM (SUMMARIZE SELECT * EXCLUDE ({', '.join(LINEAGE_COLUMNS)}) FROM {v})"
        print(f"\n--- {t} ---")
        con.sql(f"""SELECT column_name, column_type, min, max, approx_unique,
                           round(TRY_CAST(avg AS DOUBLE), 2) AS avg, q25, q50, q75,
                           null_percentage FROM ({q})""").show(max_width=200, max_rows=50)
        copy_to_csv(con, q, OUT / f"03_summary_{t}.csv")


def distinct_values(con) -> dict:
    """Value counts for categoricals. Raw vs normalised view exposes case/space variants."""
    section("4. DISTINCT VALUES (categorical columns)")
    parts = []
    for t, cols in CATEGORICAL.items():
        for c in present_columns(con, f"v_bronze_{t}", cols):   # optional columns may be absent
            parts.append(f"""
                SELECT '{t}' AS table_name, '{c}' AS column_name,
                       CAST({c} AS VARCHAR) AS raw_value, {norm(c)} AS normalised_value,
                       count(*) AS n,
                       round(100.0 * count(*) / sum(count(*)) OVER (), 3) AS pct
                FROM v_bronze_{t} GROUP BY ALL""")
    q = " UNION ALL ".join(parts) + " ORDER BY table_name, column_name, n DESC"
    con.execute(f"CREATE OR REPLACE TEMP TABLE _dv AS {q}")
    copy_to_csv(con, "SELECT * FROM _dv", OUT / "04_distinct_values.csv")

    con.sql("""SELECT table_name, column_name,
                      count(DISTINCT raw_value)        AS raw_distinct,
                      count(DISTINCT normalised_value) AS normalised_distinct,
                      sum(n) FILTER (WHERE raw_value IS NULL OR normalised_value IS NULL) AS null_or_blank,
                      string_agg(raw_value || ' (' || n || ')', ' | ' ORDER BY n DESC)
                          FILTER (WHERE raw_value IS NOT NULL)  AS top_values
               FROM _dv GROUP BY ALL ORDER BY 1, 2""").show(max_width=220, max_rows=100)

    suggested: dict = {}
    for t, c, vals in con.execute("""
            SELECT table_name, column_name,
                   list(DISTINCT normalised_value) FILTER (WHERE normalised_value IS NOT NULL)
            FROM _dv GROUP BY ALL""").fetchall():
        vals = sorted(vals or [])
        if len(vals) <= MAX_DISTINCT_FOR_ALLOWED_LIST:
            suggested.setdefault(t, {})[c] = vals
    return suggested


def time_coverage(con) -> None:
    section("5-6. TIME COVERAGE AND MONTHLY VOLUME")
    parts, month_parts = [], []
    for t in EVENT_TABLES:
        ts = EVENT_TS[t]
        parts.append(f"""
            SELECT '{t}' AS table_name, min({ts}) AS min_ts, max({ts}) AS max_ts,
                   count(*) FILTER (WHERE {ts} IS NULL)                                AS null_ts,
                   count(*) FILTER (WHERE {ts} < TIMESTAMP '{WINDOW_START}'
                                       OR {ts} >= TIMESTAMP '{WINDOW_END}' + INTERVAL 1 DAY) AS out_of_window,
                   count(*) FILTER (WHERE month IS NULL)                               AS null_month,
                   count(*) FILTER (WHERE NOT regexp_full_match(month, '\\d{{4}}-\\d{{2}}')) AS bad_month_format,
                   count(*) FILTER (WHERE month <> strftime({ts}, '%Y-%m'))            AS month_ts_mismatch
            FROM v_bronze_{t}""")
        month_parts.append(f"""
            SELECT '{t}' AS table_name, month, count(*) AS events,
                   count(DISTINCT customer_id) AS active_customers
            FROM v_bronze_{t} GROUP BY ALL""")
    q = " UNION ALL ".join(parts)
    con.sql(q).show(max_width=200)
    copy_to_csv(con, q, OUT / "05_time_coverage.csv")

    mq = " UNION ALL ".join(month_parts)
    con.sql(f"""PIVOT ({mq}) ON table_name USING sum(events) GROUP BY month ORDER BY month""").show(max_rows=40)
    copy_to_csv(con, mq + " ORDER BY table_name, month", OUT / "06_rows_by_month.csv")

    con.sql(f"""SELECT min(signup_date) AS min_signup, max(signup_date) AS max_signup,
                       count(*) FILTER (WHERE signup_date > DATE '{WINDOW_END}') AS signup_after_window,
                       count(*) FILTER (WHERE signup_date IS NULL) AS null_signup
                FROM v_bronze_customers""").show()


def fk_coverage(con) -> None:
    section("7. FOREIGN-KEY COVERAGE (event customer_id -> customers)")
    q = " UNION ALL ".join(f"""
        SELECT '{t}' AS table_name, count(*) AS rows,
               count(*) FILTER (WHERE e.customer_id IS NULL)                    AS null_customer_id,
               count(*) FILTER (WHERE e.customer_id IS NOT NULL AND c.customer_id IS NULL) AS orphan_rows,
               count(DISTINCT e.customer_id) FILTER (WHERE c.customer_id IS NULL) AS orphan_customers
        FROM v_bronze_{t} e
        LEFT JOIN (SELECT DISTINCT customer_id FROM v_bronze_customers) c USING (customer_id)"""
        for t in EVENT_TABLES)
    con.sql(q).show()
    copy_to_csv(con, q, OUT / "07_fk_coverage.csv")

    con.sql(f"""
        WITH active AS (SELECT customer_id FROM v_bronze_interactions UNION
                        SELECT customer_id FROM v_bronze_transactions UNION
                        SELECT customer_id FROM v_bronze_engagements)
        SELECT count(*) AS customers_in_profile,
               count(*) FILTER (WHERE a.customer_id IS NOT NULL) AS with_any_event,
               count(*) FILTER (WHERE a.customer_id IS NULL)     AS with_no_event
        FROM v_bronze_customers c LEFT JOIN active a USING (customer_id)""").show()


def range_preview(con) -> None:
    """How many rows would break the planned Step 3b rules (nothing is changed here)."""
    section("8. RANGE-RULE PREVIEW (rows that would be fixed / quarantined in Step 3b)")
    owner = {"age": "customers", "tenure_months": "customers", "duration_sec": "interactions",
             "csat_score": "interactions", "nps_score": "interactions", "sentiment_score_gt": "interactions",
             "session_duration_sec": "engagements", "pages_viewed": "engagements"}
    rules = [(owner[c], c, f"{c} NOT BETWEEN {lo} AND {hi}") for c, (lo, hi) in RANGES.items()]   # = Step 3b FIX rules
    rules += [
        ("interactions", "interaction_text", "interaction_text IS NULL OR length(trim(interaction_text)) = 0"),
        ("transactions", "amount", "amount IS NULL OR amount = 0"),
        ("transactions", "discount_amount", "discount_amount < 0 OR discount_amount > abs(amount)"),
        ("engagements", "session_duration_sec",
         f"session_duration_sec > {ENG_LONG_SESSION_WARN_SEC}"),          # WARN-level threshold (EDA outlier rule)
        ("interactions", "csat_score", "csat_score <> round(csat_score)"),  # non-integer ratings
        ("interactions", "nps_score", "nps_score <> round(nps_score)"),
    ]
    q = " UNION ALL ".join(f"""
        SELECT '{t}' AS table_name, '{c}' AS column_name, '{r.replace("'", "''")}' AS rule,
               count(*) FILTER (WHERE {r}) AS violating_rows,
               round(100.0 * count(*) FILTER (WHERE {r}) / count(*), 3) AS pct
        FROM v_bronze_{t}""" for t, c, r in rules)
    con.sql(q).show(max_width=200)
    copy_to_csv(con, q, OUT / "08_range_preview.csv")

    print("Amount sign by transaction_type (are refunds negative?):")
    con.sql("""SELECT transaction_type, transaction_status, count(*) AS n,
                      count(*) FILTER (WHERE amount < 0) AS negative, round(avg(amount), 2) AS avg_amount
               FROM v_bronze_transactions GROUP BY ALL ORDER BY n DESC""").show(max_rows=60)
    print("app_version populated by device_type (expected: mobile_app only):")
    con.sql("""SELECT engagement_channel, device_type, count(*) AS n,
                      count(app_version) AS app_version_populated,
                      count(campaign_id) AS campaign_populated
               FROM v_bronze_engagements GROUP BY ALL ORDER BY n DESC""").show(max_rows=60)


def per_customer_distribution(con) -> None:
    section("9. EVENTS PER CUSTOMER (whole window)")
    q = " UNION ALL ".join(f"""
        SELECT '{t}' AS table_name, count(*) AS customers, min(n) AS min,
               quantile_cont(n, 0.5) AS p50, quantile_cont(n, 0.9) AS p90,
               quantile_cont(n, 0.99) AS p99, max(n) AS max
        FROM (SELECT customer_id, count(*) AS n FROM v_bronze_{t} GROUP BY 1)"""
        for t in EVENT_TABLES)
    con.sql(q).show()
    copy_to_csv(con, q, OUT / "09_events_per_customer.csv")


def label_profile(con) -> None:
    section("10. GROUND-TRUTH LABEL PROFILE (validation only - never a feature)")
    q = """SELECT sentiment_label, count(*) AS n,
                  round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct,
                  round(min(sentiment_score_gt), 3) AS min_gt, round(avg(sentiment_score_gt), 3) AS avg_gt,
                  round(max(sentiment_score_gt), 3) AS max_gt,
                  round(avg(length(interaction_text)), 1) AS avg_text_len,
                  round(avg(csat_score), 2) AS avg_csat, round(avg(nps_score), 2) AS avg_nps
           FROM v_bronze_interactions GROUP BY ALL ORDER BY n DESC"""
    con.sql(q).show()
    copy_to_csv(con, q, OUT / "10_label_profile.csv")
    print("Populated share of optional scores:")
    con.sql("""SELECT round(100.0 * count(csat_score) / count(*), 1) AS csat_pct_populated,
                      round(100.0 * count(nps_score)  / count(*), 1) AS nps_pct_populated
               FROM v_bronze_interactions""").show()


def samples(con) -> None:
    for t in TABLES:
        copy_to_csv(con, f"SELECT * FROM v_bronze_{t} USING SAMPLE 20 ROWS",
                    OUT / f"11_samples_{t}.csv")
    section("11. SAMPLE ROWS (5 per table; 20 saved to CSV)")
    for t in TABLES:
        print(f"\n--- {t} ---")
        con.sql(f"SELECT * EXCLUDE ({', '.join(LINEAGE_COLUMNS)}) FROM v_bronze_{t} LIMIT 5"
                ).show(max_width=220)

# ---------------------------------------------------------------------------

## 4. Main

In [ ]:
def main() -> bool:
    OUT.mkdir(parents=True, exist_ok=True)
    CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    con = get_connection()
    try:
        register_layer(con, BRONZE_DIR, "v_bronze")
        overview(con)
        schema_and_summary(con)
        suggested = distinct_values(con)
        time_coverage(con)
        fk_coverage(con)
        range_preview(con)
        per_customer_distribution(con)
        label_profile(con)
        samples(con)
    finally:
        con.close()
    ALLOWED_VALUES_SUGGESTED.write_text(json.dumps(suggested, indent=2))
    log(f"Profile reports written to {OUT}")
    log(f"Suggested allowed values written to {ALLOWED_VALUES_SUGGESTED}")
    log("NEXT: review the suggested list against the data dictionary, remove any invalid "
        "values, and save it as config/allowed_values.json before running Step 3b.")
    return True

In [ ]:
if __name__ == "__main__":
    finish(main())